In [ ]:
import torch
import torch.nn as nn

from torchvision import datasets, transforms
from torch.utils.data import DataLoader,TensorDataset

In [ ]:
transform = transforms.ToTensor()

train = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform = transform
)

test = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform = transform
)


train_loader = DataLoader(
    train,
    batch_size=128,
    shuffle = True
)

test_loader = DataLoader(

    test,

    batch_size=128,

    shuffle=False

)

In [ ]:
class Maxout(nn.Module):

  def __init__(self, in_features, out_features, pool_size):

    super().__init__()

    self.linear = nn.Linear(in_features, out_features*pool_size)

    self.in_feartures = in_features

    self.out_features = out_features

    self.pool_size = pool_size

  def forward(self,x):

    x = self.linear(x)

    x = x.view(x.size(0), self.out_features, self.pool_size)

    return x.max(dim=2).values

In [ ]:
class MaxoutNet(nn.Module):

  def __init__(self,pool_size):
    super().__init__()

    self.maxout1 = Maxout(in_features=784,out_features=240,pool_size=pool_size)
    self.maxout2 = Maxout(in_features=240,out_features=240,pool_size=pool_size)
    self.output = nn.Linear(240,10)

  def forward(self,x):

    x = x.view(x.size(0), -1)
    x = self.maxout1(x)
    x = self.maxout2(x)
    x = self.output(x)

    return x


In [ ]:
model = MaxoutNet(pool_size=5)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.001
)

epochs = 5

for epoch in range(epochs):

  model.train()
  total_loss = 0

  for x,y in train_loader:

    optimizer.zero_grad()

    output = model(x)

    loss = criterion(output,y)

    loss.backward()

    optimizer.step()

    total_loss += loss.item()

  avg_loss = total_loss/len(train_loader)

  print(f"Epoch {epoch + 1}, loss = {avg_loss:.4f}")


In [ ]:
with torch.no_grad():

  total = 0
  correct = 0

  for x,y in test_loader:

    output = model(x)
    _,predicted = torch.max(output,dim=1)
    total += y.size(0)
    correct += (predicted == y).sum().item()

accuracy = correct / total

print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
torch.save(model.state_dict(), "maxout_mnist.pth")

***Generating Adversary Examples & Accuracy Test***

In [ ]:
epsilon = 0.07

adv_x = []

adv_y = []

model.eval()

for x, y in test:

    x = x.unsqueeze(0)

    y = torch.tensor([y])

    x.requires_grad_()

    output = model(x)

    loss = criterion(output, y)

    model.zero_grad()

    loss.backward()

    nabla = x.grad

    attack = epsilon * nabla.sign()

    x_adv = x + attack

    adv_x.append(x_adv.detach().squeeze(0))

    adv_y.append(y.squeeze(0))

adv_x = torch.stack(adv_x)

adv_y = torch.stack(adv_y)

adv_test = TensorDataset(adv_x, adv_y)

adv_test_loader = DataLoader(

    adv_test,

    batch_size=128,

    shuffle=False

)

total = 0
correct = 0

for x,y in adv_test_loader:

  output = model(x)
  _,predicted = torch.max(output,dim=1)
  total += y.size(0)
  correct += (predicted == y).sum().item()

accuracy = correct / total

print(f"Test Accuracy: {accuracy:.4f}")

***Adversary Training***

In [ ]:
def fgsm_attack(model, criterion, x, y, epsilon):
    x = x.clone().detach()
    x.requires_grad_()

    output = model(x)
    loss = criterion(output, y)

    model.zero_grad()
    loss.backward()

    attack = epsilon * x.grad.sign()

    x_adv = x + attack

    return x_adv.detach()

In [ ]:
epsilon = 0.07

adv_train_x = []
adv_train_y = []

for x, y in train:

    x = x.unsqueeze(0)
    y = torch.tensor([y])

    x_adv = fgsm_attack(
        model,
        criterion,
        x,
        y,
        epsilon
    )

    adv_train_x.append(x_adv.squeeze(0))
    adv_train_y.append(y.squeeze(0))

adv_train_x = torch.stack(adv_train_x)
adv_train_y = torch.stack(adv_train_y)

adv_train = TensorDataset(adv_train_x, adv_train_y)

adv_train_loader = DataLoader(
    adv_train,
    batch_size=128,
    shuffle=True
)

In [ ]:
model = MaxoutNet(pool_size=5)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001)

epochs = 5

for epoch in range(epochs):

  model.train()
  total_loss = 0

  for x,y in adv_train_loader:

    optimizer.zero_grad()

    output = model(x)

    loss = criterion(output,y)

    loss.backward()

    optimizer.step()

    total_loss += loss.item()

  avg_loss = total_loss/len(adv_train_loader)

  print(f"Epoch {epoch + 1}, loss = {avg_loss:.4f}")

***Testing New Model***

In [ ]:
adv_test_x = []
adv_test_y = []

for x, y in test:

    x = x.unsqueeze(0)
    y = torch.tensor([y])

    x_adv = fgsm_attack(
        model,
        criterion,
        x,
        y,
        epsilon
    )

    adv_test_x.append(x_adv.squeeze(0))
    adv_test_y.append(y.squeeze(0))

adv_test_x = torch.stack(adv_test_x)
adv_test_y = torch.stack(adv_test_y)

adv_test = TensorDataset(
    adv_test_x,
    adv_test_y
)

adv_test_loader = DataLoader(
    adv_test,
    batch_size=128,
    shuffle=False
)

In [ ]:
for x,y in adv_test_loader:

  output = model(x)
  _,predicted = torch.max(output,dim=1)
  total += y.size(0)
  correct += (predicted == y).sum().item()

accuracy = correct / total

print(f"Test Accuracy: {accuracy:.4f}")